# DynaDetect-WM Colab Runner

This notebook is a lightweight launcher for the existing Python scripts in this repository.

Use it to:
- set up a Colab runtime
- clone or reuse the repo
- install dependencies
- optionally configure a Kaggle legacy API key with `kaggle.json`
- optionally unzip or download large datasets
- run training and phase scripts with a small number of editable variables


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/dd2-wm.git'
PROJECT_DIR = '/content/dd2-wm'
USE_GIT_CLONE = True

if USE_GIT_CLONE:
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print(os.getcwd())


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '--version'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kaggle'], check=True)


In [ ]:
import os
import shutil
import subprocess

os.makedirs('data', exist_ok=True)

USE_KAGGLE_API = False
KAGGLE_JSON_DRIVE_PATH = '/content/drive/MyDrive/kaggle.json'
DOWNLOAD_VGGFACE2_FROM_KAGGLE = False
DOWNLOAD_CHEXPERT_FROM_KAGGLE = False
VGGFACE2_KAGGLE_DATASET = 'OWNER/VGGFACE2_DATASET'
CHEXPERT_KAGGLE_DATASET = 'OWNER/CHEXPERT_DATASET'
COPY_VGGFACE2 = False
COPY_CHEXPERT = False
VGGFACE2_ZIP = '/content/drive/MyDrive/VGGFace2.zip'
CHEXPERT_ZIP = '/content/drive/MyDrive/CheXpert.zip'

def extract_zip(zip_path, dest_dir):
    subprocess.run(['unzip', '-q', zip_path, '-d', dest_dir], check=True)

def download_kaggle_dataset(dataset_slug, dest_dir):
    subprocess.run(['mkdir', '-p', '/root/.kaggle'], check=True)
    shutil.copy2(KAGGLE_JSON_DRIVE_PATH, '/root/.kaggle/kaggle.json')
    subprocess.run(['chmod', '600', '/root/.kaggle/kaggle.json'], check=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', dest_dir, '--unzip'], check=True)

if USE_KAGGLE_API and DOWNLOAD_VGGFACE2_FROM_KAGGLE:
    download_kaggle_dataset(VGGFACE2_KAGGLE_DATASET, '/content/dd2-wm/data/')

if USE_KAGGLE_API and DOWNLOAD_CHEXPERT_FROM_KAGGLE:
    download_kaggle_dataset(CHEXPERT_KAGGLE_DATASET, '/content/dd2-wm/data/')

if COPY_VGGFACE2:
    local_zip = '/content/dd2-wm/data/VGGFace2.zip'
    shutil.copy2(VGGFACE2_ZIP, local_zip)
    extract_zip(local_zip, '/content/dd2-wm/data/')

if COPY_CHEXPERT:
    local_zip = '/content/dd2-wm/data/CheXpert.zip'
    shutil.copy2(CHEXPERT_ZIP, local_zip)
    extract_zip(local_zip, '/content/dd2-wm/data/')

print(sorted(os.listdir('data')))


In [ ]:
DATASET = 'cifar100'
EPOCHS = 1
BATCH_SIZE = 128
NUM_POISONS = 10
PHASE4_EPOCHS = 1

if DATASET in ['vggface', 'chexpert']:
    BATCH_SIZE = 64

print({
    'dataset': DATASET,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'num_poisons': NUM_POISONS,
    'phase4_epochs': PHASE4_EPOCHS
})


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    'train.py',
    '--dataset', DATASET,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
], check=True)


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    'run_phase2.py',
    '--dataset', DATASET,
    '--num-poisons', str(NUM_POISONS),
    '--model-path', f'checkpoints/best_{DATASET}_resnet18.pth',
], check=True)

subprocess.run([
    sys.executable,
    'run_phase3.py',
    '--dataset', DATASET,
    '--num-poisons', str(NUM_POISONS),
    '--model-path', f'checkpoints/best_{DATASET}_resnet18.pth',
], check=True)

subprocess.run([
    sys.executable,
    'run_phase4.py',
    '--dataset', DATASET,
    '--num-poisons', str(NUM_POISONS),
    '--epochs', str(PHASE4_EPOCHS),
    '--auth-model-path', f'checkpoints/best_{DATASET}_resnet18.pth',
], check=True)


In [ ]:
import os
import shutil
from pathlib import Path

BACKUP_DIR = '/content/drive/MyDrive/dd2-wm-backups'
os.makedirs(BACKUP_DIR, exist_ok=True)

if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', os.path.join(BACKUP_DIR, 'checkpoints'), dirs_exist_ok=True)

if os.path.exists('results'):
    shutil.copytree('results', os.path.join(BACKUP_DIR, 'results'), dirs_exist_ok=True)

for path in sorted(Path('results').glob('*')):
    print(path)
